BIAS HUNTER FINAL PROJECT
Combining RAG + AUTOMATED LLM DATAOPS

This project demonstrates TWO core techniques:

TECHNIQUE 1: RAG (Retrieval-Augmented Generation)      
  - Retrieves relevant polling data from a CSV corpus    
  - Computes poll-implied probabilities                  
  - Provides grounded evidence for decision-making  
Option 2: AGENT                                     
  - Uses Kalshi API tool to fetch live market data       
  - Compares market prices against fair probabilities    
  - Makes threshold-based classification decisions  
TECHNIQUE 2: AUTOMATED LLM DATAOPS                    
  - LLM Labeler: Auto-labels data with sentiment/risk  
  - LLM Judge: Evaluates market efficiency               
  - Generates structured outputs for downstream use      


## IMPORTS + SET UP
* chat gpt and claude ai were used to help clean and write code

In [ ]:
# IMPORTS

!pip install -U langchain==0.3.27 langchain-openai==0.3.34 langchain-azure-ai==0.1.6 langchain-community==0.3.30 langchain-text-splitters==0.3.11 faiss-cpu==1.12.0 pypdf==6.1.1 python-docx==1.2.0

import os
import json
import re
from typing import Dict, List, Optional, Any
from datetime import datetime

import requests
import pandas as pd

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
import textwrap

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.8/52.8 kB 4.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 4.3 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.8/82.8 kB 4.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.1/47.1 kB 3.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 110.3/110.3 kB 9.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.5/88.5 kB 7.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.6/74.6 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 45.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.6/75.6 kB 6.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 85.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 31.4/31.4 MB 38.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/

In [ ]:
# ============================================================
# CONFIGURATION - Azure LLM Setup
# ============================================================
# These settings connect to Azure's AI inference API.
# SIMPLIFIED: Only requires AZURE_API_KEY - used for everything!
#
# The LLM is used by all three techniques:
#   - RAG: Formats output (optional)
#   - AGENT: Reasons about market data and generates recommendations
#   - DATAOPS: Performs labeling and judging tasks
# ============================================================

# Your class identifier for the Azure endpoint
CLASS = "MIS372T"

# Set this in your notebook before running:
AZURE_API_KEY = "5aa2f6d151e04a2a92023c0d0a74b355"

# Azure endpoint URL
AZURE_INFERENCE_BASE = f"https://aistudio-apim-ai-gateway02.azure-api.net/{CLASS}/v1/models"

# Model configuration
CHAT_MODEL_NAME = "gpt-4.1-nano"
CHAT_API_VERSION = "2024-05-01-preview"

# Headers for Azure API Management - uses same AZURE_API_KEY
APIM_HEADERS = (
    {"Ocp-Apim-Subscription-Key": AZURE_API_KEY}
    if AZURE_API_KEY else None
)


def build_llm():
    """
    Initialize the Azure LLM for use across all techniques.

    This function creates a LangChain-compatible chat model that:
    - Connects to Azure AI inference endpoint
    - Uses AZURE_API_KEY for both authentication and subscription
    - Can be used by Agents, Labelers, and Judges

    Returns:
        ChatModel: A LangChain chat model instance

    Raises:
        RuntimeError: If AZURE_API_KEY is not set
    """
    # Check that AZURE_API_KEY is set
    if not AZURE_API_KEY:
        raise RuntimeError(
            "Missing AZURE_API_KEY!\n"
            "Set it in your notebook: AZURE_API_KEY = 'your-key-here'\n"
            "Or as environment variable: export AZURE_API_KEY='your-key-here'"
        )

    # Create the LLM - uses AZURE_API_KEY for both credential and headers
    llm = init_chat_model(
        model=CHAT_MODEL_NAME,
        model_provider="azure_ai",
        endpoint=AZURE_INFERENCE_BASE,
        credential=AZURE_API_KEY,  # Same key for credential
        api_version=CHAT_API_VERSION,
        client_kwargs={"headers": APIM_HEADERS},  # Same key in headers
    )
    return llm


## TECHNIQUE 1: RAG (Retrieval-Augmented Generation)

In [ ]:
class PollingRAG:
    """
    ┌─────────────────────────────────────────────────────────┐
    │                    TECHNIQUE 1: RAG                     │
    │         Retrieval-Augmented Generation for Polls        │
    └─────────────────────────────────────────────────────────┘

    This class implements RAG over a polling dataset:

    STEP 1 (INDEXING): Load polling CSV into memory
        - Parse dates for temporal filtering
        - Validate required columns exist

    STEP 2 (RETRIEVAL): Given a race_id and date, retrieve relevant polls
        - Filter by race_id (exact match)
        - Filter by date (polls on or before as_of_date)
        - Sort by recency, take top_k most recent

    STEP 3 (GENERATION): Compute poll-implied probability
        - Average the support percentages from retrieved polls
        - Return probability + evidence for transparency
    """

    def __init__(self, csv_path: str, top_k: int = 5):
        """
        Initialize the RAG system by loading the polling corpus.

        This is the INDEXING step of RAG - we load and prepare
        the document corpus for later retrieval.

        Args:
            csv_path (str): Path to the polling CSV file
            top_k (int): How many recent polls to retrieve (default: 5)

        Required CSV columns:
            - snapshot_date: When the poll was conducted
            - race_id: Identifier for the race (e.g., "2024_AZ_SEN")
            - candidate1_support_pct: Support percentage for candidate 1
        """
        # ---- RAG STEP 1: INDEXING ----
        # Load the document corpus (polling data) into memory
        self.df = pd.read_csv(csv_path)
        self.top_k = top_k

        # Validate that required columns exist
        if "snapshot_date" not in self.df.columns:
            raise ValueError("Polling CSV missing 'snapshot_date' column.")
        if "race_id" not in self.df.columns:
            raise ValueError("Polling CSV missing 'race_id' column.")

        # Convert dates for temporal filtering during retrieval
        self.df["snapshot_date"] = pd.to_datetime(self.df["snapshot_date"])

    def retrieve_polls(
        self,
        race_id: str,
        as_of_date: Optional[str] = None,
    ) -> pd.DataFrame:
        """
        ┌─────────────────────────────────────────────────────┐
        │              RAG STEP 2: RETRIEVAL                  │
        └─────────────────────────────────────────────────────┘

        Retrieve the most relevant polls for a given race.

        This is the core RETRIEVAL step of RAG:
        1. Filter by race_id (semantic matching)
        2. Filter by date (temporal relevance)
        3. Sort by recency (most recent = most relevant)
        4. Return top_k results

        Args:
            race_id (str): The race to query (e.g., "2024_AZ_SEN")
            as_of_date (str, optional): Only include polls on/before this date

        Returns:
            pd.DataFrame: The top_k most relevant polls
        """
        # Filter 1: Match the race_id exactly
        sub = self.df[self.df["race_id"] == race_id].copy()

        if sub.empty:
            return sub  # No polls found for this race

        # Filter 2: Temporal filtering - only polls before cutoff date
        if as_of_date is not None:
            cutoff = pd.to_datetime(as_of_date)
            sub = sub[sub["snapshot_date"] <= cutoff]

        # Sort by recency and take top_k (most recent polls are most relevant)
        sub = sub.sort_values("snapshot_date", ascending=False).head(self.top_k)

        return sub

    def compute_poll_implied_prob(
        self,
        race_id: str,
        as_of_date: Optional[str] = None,
        candidate_support_col: str = "candidate1_support_pct",
    ):
        """
        ┌─────────────────────────────────────────────────────┐
        │              RAG STEP 3: GENERATION                 │
        └─────────────────────────────────────────────────────┘

        Generate a poll-implied probability from retrieved evidence.

        This combines retrieval results into a single probability:
        1. Call retrieve_polls() to get relevant polls
        2. Extract support percentages from retrieved polls
        3. Average them to get poll-implied probability
        4. Return BOTH the probability AND the evidence (for citations)

        Args:
            race_id (str): The race to query
            as_of_date (str, optional): Date cutoff for polls
            candidate_support_col (str): Column name for support percentage

        Returns:
            tuple: (poll_probability, list_of_evidence_dicts)
                - poll_probability: float in [0, 1] or None if no data
                - evidence: List of poll records used (for transparency)
        """
        # First, RETRIEVE the relevant polls
        polls = self.retrieve_polls(race_id, as_of_date)

        # Handle case where no polls are found
        if polls.empty or candidate_support_col not in polls.columns:
            return None, []

        # GENERATE: Compute average probability from retrieved polls
        # Convert from percentage (0-100) to probability (0-1)
        probs = polls[candidate_support_col].astype(float) / 100.0
        poll_prob = float(probs.mean())

        # Prepare evidence for transparency/citation
        # This is key for RAG - we return the sources we used!
        cols_to_keep = [
            "race_id",
            "snapshot_date",
            "pollster",
            "sample_size",
            "candidate1_name",
            "candidate1_support_pct",
            "candidate2_name",
            "candidate2_support_pct",
        ]
        cols_to_keep = [c for c in cols_to_keep if c in polls.columns]
        evidence = polls[cols_to_keep].to_dict(orient="records")

        return poll_prob, evidence

## TECHNIQUE 2: AGENT


In [ ]:
class KalshiMarketDataTool:
    """
    This is the TOOL that the Agent uses to interact with the
    external world (Kalshi prediction markets).

    WHAT IS A TOOL?
    - A tool is a function/class that an Agent can call
    - It interfaces with external APIs, databases, or services
    - The Agent decides WHEN to call the tool and HOW to use results

    KALSHI API ENDPOINTS USED:
    - GET /series/{ticker} - Get series info
    - GET /markets - Get market data
    - GET /markets/{ticker}/orderbook - Get orderbook depth
    """

    def __init__(self, base_url: str = "https://api.elections.kalshi.com/trade-api/v2"):
        """
        Initialize the Kalshi API tool.

        Args:
            base_url (str): Base URL for Kalshi's public API
        """
        self.base_url = base_url.rstrip("/")


    def get_markets_by_ticker_list(self, tickers: List[str]) -> List[Dict]:
        """
        TOOL METHOD: Get specific markets by their tickers.

        Args:
            tickers (List[str]): List of market tickers to fetch

        Returns:
            List[Dict]: List of market objects
        """
        if not tickers:
            return []

        url = f"{self.base_url}/markets"
        params = {"tickers": ",".join(tickers)}
        resp = requests.get(url, params=params, timeout=10)
        resp.raise_for_status()
        data = resp.json()
        return data.get("markets", [])


    def get_market_price_snapshot(self, market_ticker: str) -> Optional[Dict]:
        """
        TOOL METHOD: Get a simplified price snapshot for a market.

        This is the primary method used by the Agent to get
        current market state.

        Args:
            market_ticker (str): The market to query

        Returns:
            Dict or None: Simplified market snapshot with:
                - market_ticker: The ticker string
                - title: Human-readable market title
                - yes_price_pct: Market-implied probability (0-100)
                - volume: Trading volume
        """
        markets = self.get_markets_by_ticker_list([market_ticker])
        if not markets:
            return None

        m = markets[0]

      # 1) Try yes_price
        yes_price_cents = m.get("yes_price")

    # 2) Fallback: last_price
        if yes_price_cents is None:
            yes_price_cents = m.get("last_price")

    # 3) Fallback: mid of yes_bid/yes_ask if both exist
        if yes_price_cents is None:
            yes_bid = m.get("yes_bid")
            yes_ask = m.get("yes_ask")
            if yes_bid is not None and yes_ask is not None:
                yes_price_cents = (yes_bid + yes_ask) / 2.0

        volume = m.get("volume")

        yes_price_pct = float(yes_price_cents) if yes_price_cents is not None else None

        return {
            "market_ticker": m.get("ticker"),
            "title": m.get("title"),
            "event_ticker": m.get("event_ticker"),
            "series_ticker": m.get("series_ticker"),
            "yes_price_pct": yes_price_pct,
            "volume": volume,
            "raw": m,
        }

## TECHNIQUE 3: AUTOMATED LLM DATAOPS

In [ ]:
LABELING_PROMPT = ChatPromptTemplate.from_template(
    """

DATAOPS: LABELING TASK

You are a financial data labeling assistant. Your job is to
analyze market/polling data and assign structured labels.

DATA TO LABEL:
{data_point}

LABELING INSTRUCTIONS:
Analyze the data and provide labels as JSON with these exact fields:

1. sentiment: Assess the overall market sentiment
   - "bullish" = market/polls suggest positive outcome
   - "neutral" = no clear direction
   - "bearish" = market/polls suggest negative outcome

2. time_horizon: How soon is the event?
   - "immediate" = within days
   - "near_term" = within weeks
   - "long_term" = months or more

3. confidence_level: How reliable is the data?
   - "high" = large sample, consistent data
   - "medium" = moderate confidence
   - "low" = small sample, inconsistent, or stale data

4. risk_flags: List any concerns (choose all that apply):
   - "polling_bias" = polls may be systematically biased
   - "low_volume" = insufficient market liquidity
   - "high_volatility" = prices are unstable
   - "stale_data" = data may be outdated
   - "none" = no significant risks identified

5. reasoning: One sentence explaining your labels

Return ONLY valid JSON, no other text.
"""
)


class LLMDataLabeler:
    """
    TECHNIQUE #a: DATAOPS LABELER

    This class uses an LLM to automatically label data points
    with structured metadata.

    LABELS GENERATED:
    - sentiment: bullish/neutral/bearish
    - time_horizon: immediate/near_term/long_term
    - confidence_level: high/medium/low
    - risk_flags: list of identified risks
    - reasoning: explanation of labels
    """

    def __init__(self):
        """Initialize the labeler with an LLM."""
        self.llm = build_llm()
        self.parser = StrOutputParser()

    def label(self, data_point: Dict) -> Dict:
        """
        Label a single data point with structured metadata.

        WORKFLOW:
        1. Format data point as JSON string
        2. Send to LLM with labeling prompt
        3. Parse LLM response as JSON
        4. Return structured labels

        Args:
            data_point (Dict): The data to label

        Returns:
            Dict: Structured labels
        """
        chain = LABELING_PROMPT | self.llm | self.parser

        result = chain.invoke({
            "data_point": json.dumps(data_point, indent=2, default=str)
        })

        try:
            cleaned = re.sub(r'^```json\s*|\s*```$', '', result.strip())
            return json.loads(cleaned)
        except json.JSONDecodeError:
            return {"raw_response": result, "parse_error": True}



# ============================================================
# DATAOPS COMPONENT 2: LLM JUDGE
# ============================================================

JUDGE_PROMPT = ChatPromptTemplate.from_template(
    """

You are an expert judge evaluating prediction market efficiency.
Your job is to assess whether the market price reflects fair value.

MARKET DATA TO EVALUATE:
- Market Probability: {market_prob}
- Poll Probability: {poll_prob}
- Discrepancy: {discrepancy}
- Trading Volume: {volume}

EVALUATION CRITERIA:
1. Is the discrepancy significant enough to suggest mispricing?
2. Does the volume indicate sufficient liquidity?
3. Are there signs of market inefficiency or bias?
4. How reliable is the polling data as a baseline?

Provide your judgment as JSON with these fields:

1. efficiency_score: float 0-10 (10 = perfectly efficient)

2. verdict: one of:
   - "efficient"
   - "slightly_inefficient"
   - "inefficient"
   - "highly_inefficient"

3. mispricing_direction: one of:
   - "overpriced" (market > fair)
   - "underpriced" (market < fair)
   - "fair")
   - "uncertain"

4. confidence: float 0-1

5. key_insights: list of 2-3 short observations

6. recommendation: one sentence suggestion

Return ONLY valid JSON.
"""
)


class LLMJudge:
    """
    This class uses an LLM to judge/evaluate data quality
    and market efficiency.

    OUTPUTS:
    - efficiency_score: 0-10 rating
    - verdict: efficiency classification
    - mispricing_direction: over/under/fair
    - key_insights: observations
    - recommendation: actionable suggestion
    """

    def __init__(self):
        """Initialize the judge with an LLM."""
        self.llm = build_llm()
        self.parser = StrOutputParser()

    def judge_efficiency(self, market_prob: float, poll_prob: float, volume: int = 0) -> Dict:
        """
        Judge market efficiency by comparing market vs poll probabilities.

        WORKFLOW:
        1. Calculate discrepancy
        2. Format data for LLM
        3. LLM evaluates and provides structured judgment
        4. Parse and return judgment

        Args:
            market_prob (float): Market-implied probability
            poll_prob (float): Poll-implied probability
            volume (int): Trading volume

        Returns:
            Dict: Structured judgment with score, verdict, insights
        """
        # Calculate discrepancy
        # Ensure discrepancy is None if either market_prob or poll_prob is None
        discrepancy = None
        if market_prob is not None and poll_prob is not None:
            discrepancy = market_prob - poll_prob

        chain = JUDGE_PROMPT | self.llm | self.parser

        result = chain.invoke({
            "market_prob": f"{market_prob:.2%}" if market_prob is not None else "N/A",
            "poll_prob": f"{poll_prob:.2%}" if poll_prob is not None else "N/A",
            "discrepancy": f"{discrepancy:.2%}" if discrepancy is not None else "N/A",
            "volume": volume,
        })

        try:
            cleaned = re.sub(r'^```json\s*|\s*```$', '', result.strip())
            return json.loads(cleaned)
        except json.JSONDecodeError:
            return {"raw_response": result, "parse_error": True}

In [ ]:
# ============================================================
# ============================================================
#
#           FULL INTEGRATED SYSTEM: RAG + AGENT + DATAOPS
#
# ============================================================
# ============================================================


class BiasHunterWithDataOps:
    """
    This class integrates all three techniques into one pipeline:

    1. RAG: Retrieve polling data and compute poll probability
    2. AGENT: Fetch market data via Kalshi tool
    3. DATAOPS:
       a. Label the combined data (sentiment, risk flags)
       b. Judge market efficiency (score, verdict, insights)
    4. Generate final verdict and report
    """

    def __init__(self, polling_csv_path: str):
        """
        Initialize all three technique components.

        Args:
            polling_csv_path (str): Path to polling CSV for RAG
        """
        # ---- TECHNIQUE 1: RAG ----
        self.poll_rag = PollingRAG(csv_path=polling_csv_path)

        # ---- TECHNIQUE 2: AGENT TOOL ----
        self.kalshi = KalshiMarketDataTool()

        # ---- TECHNIQUE 3: DATAOPS ----
        self.labeler = LLMDataLabeler()
        self.judge = LLMJudge()

        # LLM for any additional reasoning
        self.llm = build_llm()
        self._parser = StrOutputParser()

    def full_analysis_with_dataops(
        self,
        market_ticker: str,
        race_id: str,
        as_of_date: Optional[str] = None,
        threshold: float = 0.10,
    ) -> Dict:
        """

        PIPELINE:
        1. [RAG] Retrieve polling data → poll_prob + evidence
        2. [AGENT] Fetch market data → market_prob + volume
        3. [DATAOPS-LABEL] Auto-label combined data → sentiment, risks
        4. [DATAOPS-JUDGE] Judge efficiency → score, verdict, insights
        5. [OUTPUT] Compute final verdict and return structured results

        Args:
            market_ticker (str): Kalshi market ticker
            race_id (str): Race ID for polling lookup
            as_of_date (str, optional): Date cutoff for polls
            threshold (float): Discrepancy threshold

        Returns:
            Dict: Complete analysis with all technique outputs
        """
        # Initialize results dictionary
        results = {
            "market_ticker": market_ticker,
            "race_id": race_id,
            "threshold": threshold,
            "timestamp": datetime.now().isoformat(),
        }

        # ════════════════════════════════════════════════════
        # TECHNIQUE 1: RAG - Retrieve Polling Data
        # ════════════════════════════════════════════════════

        poll_prob, evidence = self.poll_rag.compute_poll_implied_prob(
            race_id=race_id,
            as_of_date=as_of_date,
        )

        results["rag_results"] = {
            "technique": "RAG (Retrieval-Augmented Generation)",
            "description": "Retrieved relevant polls and computed probability",
            "poll_implied_prob": poll_prob,
            "num_polls_retrieved": len(evidence),
            "polling_evidence": evidence,
        }


        # ════════════════════════════════════════════════════
        # IMEDIATOR: AGENT - Fetch Market Data via Tool
        # ════════════════════════════════════════════════════

        snapshot = self.kalshi.get_market_price_snapshot(market_ticker)

        if snapshot is None:
            results["agent_results"] = {
                "technique": "AGENT (Tool-based)",
                "error": f"No market found for {market_ticker}"
            }
            market_prob = None
            volume = 0
        else:
            market_prob = (
                snapshot["yes_price_pct"] / 100.0
                if snapshot.get("yes_price_pct") is not None
                else None
            )
            volume = snapshot.get("volume", 0)

            results["agent_results"] = {
                "technique": "AGENT (Tool-based)",
                "description": "Used Kalshi API tool to fetch live market data",
                "market_ticker": snapshot.get("market_ticker"),
                "title": snapshot.get("title"),
                "market_prob": market_prob,
                "yes_price_pct": snapshot.get("yes_price_pct"),
                "volume": volume,
            }

        # ════════════════════════════════════════════════════
        # TECHNIQUE 2: DATAOPS - Auto-Label the Data
        # ════════════════════════════════════════════════════

        # Prepare combined data for labeling
        combined_data = {
            "market_prob": market_prob,
            "poll_prob": poll_prob,
            "volume": volume,
            "num_polls": len(evidence),
            "discrepancy": (market_prob - poll_prob) if (market_prob is not None and poll_prob is not None) else None,
        }

        labels = self.labeler.label(combined_data)

        results["dataops_labels"] = {
            "technique": "DATAOPS (LLM-based Labeling)",
            "description": "LLM automatically labeled data with sentiment and risk flags",
            "labels": labels,
        }


        # ════════════════════════════════════════════════════
        # TECHNIQUE 2b: DATAOPS - Judge Market Efficiency
        # ════════════════════════════════════════════════════

        if market_prob is not None:
            judgment = self.judge.judge_efficiency(
                market_prob=market_prob,
                poll_prob=poll_prob,
                volume=volume,
            )

            results["dataops_judgment"] = {
                "technique": "DATAOPS (LLM-as-Judge)",
                "description": "LLM evaluated market efficiency vs polling baseline",
                "judgment": judgment,
            }

        else:
            results["dataops_judgment"] = {
                "technique": "DATAOPS (LLM-as-Judge)",
                "error": "Cannot judge - missing market probability"
            }

        # ════════════════════════════════════════════════════
        # FINAL VERDICT
        # ════════════════════════════════════════════════════

        if market_prob and poll_prob:
            discrepancy = market_prob - poll_prob

            if abs(discrepancy) < threshold:
                verdict = "FAIRLY_PRICED"
            elif discrepancy > 0:
                verdict = "OVERPRICED"
            else:
                verdict = "UNDERPRICED"

            results["final_verdict"] = verdict
            results["discrepancy"] = discrepancy
            results["discrepancy_pct"] = f"{discrepancy:.2%}"

        else:
            results["final_verdict"] = "INSUFFICIENT_DATA"

        return results

    def generate_report(self, analysis: Dict) -> str:
        """
        Convert the analysis dict into a formatted report.

        Args:
            analysis (Dict): Output from full_analysis_with_dataops()

        Returns:
            str: Formatted report string
        """
        report = []

        report.append("=" * 70)
        report.append("                    BIAS HUNTER ANALYSIS REPORT")
        report.append("              RAG + AGENT + AUTOMATED LLM DATAOPS")
        report.append("=" * 70)

        report.append(f"\nMarket Ticker: {analysis.get('market_ticker')}")
        report.append(f"Race ID: {analysis.get('race_id')}")
        report.append(f"Threshold: {analysis.get('threshold')}")

        # RAG Results
        report.append("\n" + "-" * 70)
        report.append(" TECHNIQUE 1: RAG (Retrieval-Augmented Generation)")
        report.append("-" * 70)
        rag = analysis.get("rag_results", {})
        report.append(f"Poll-Implied Probability: {rag.get('poll_implied_prob', 'N/A')}")
        report.append(f"Number of Polls Retrieved: {rag.get('num_polls_retrieved', 0)}")
        if rag.get("polling_evidence"):
            report.append("Evidence Summary:")
            for i, poll in enumerate(rag.get("polling_evidence", [])[:3], 1):
                pollster = poll.get('pollster', 'Unknown')
                support = poll.get('candidate1_support_pct', 'N/A')
                report.append(f"  {i}. {pollster}: {support}%")

        # Agent Results
        report.append("\n" + "-" * 70)
        report.append(" INTERMEDIATE: AGENT (Kalshi Market Tool)")
        report.append("-" * 70)
        agent = analysis.get("agent_results", {})
        report.append(f"Market Title: {agent.get('title', 'N/A')}")
        report.append(f"Market Probability: {agent.get('market_prob', 'N/A')}")
        report.append(f"Trading Volume: {agent.get('volume', 'N/A')}")

        # DataOps Labels
        report.append("\n" + "-" * 70)
        report.append("  TECHNIQUE 2a: DATAOPS (LLM Labeler)")
        report.append("-" * 70)
        labels = analysis.get("dataops_labels", {}).get("labels", {})
        report.append(f"Sentiment: {labels.get('sentiment', 'N/A')}")
        report.append(f"Time Horizon: {labels.get('time_horizon', 'N/A')}")
        report.append(f"Confidence Level: {labels.get('confidence_level', 'N/A')}")
        report.append(f"Risk Flags: {labels.get('risk_flags', [])}")
        reasoning_text = labels.get('reasoning', 'N/A')
        wrapped_reasoning = textwrap.fill(reasoning_text, width=65, initial_indent="Reasoning: ", subsequent_indent="           ")
        report.append(wrapped_reasoning)

        # DataOps Judgment
        report.append("\n" + "-" * 70)
        report.append("  TECHNIQUE 2b: DATAOPS (LLM Judge)")
        report.append("-" * 70)
        judgment = analysis.get("dataops_judgment", {}).get("judgment", {})
        report.append(f"Efficiency Score: {judgment.get('efficiency_score', 'N/A')}/10")
        report.append(f"Verdict: {judgment.get('verdict', 'N/A')}")
        report.append(f"Mispricing Direction: {judgment.get('mispricing_direction', 'N/A')}")
        report.append(f"Confidence: {judgment.get('confidence', 'N/A')}")
        if judgment.get("key_insights"):
            report.append("Key Insights:")
            for insight in judgment.get("key_insights", []):
                report.append(f"  • {insight}")
        report.append(f"Recommendation: {judgment.get('recommendation', 'N/A')}")

        # Final Verdict
        report.append("\n" + "=" * 70)
        report.append(" FINAL VERDICT")
        report.append("=" * 70)
        report.append(f"Discrepancy: {analysis.get('discrepancy_pct', 'N/A')}")
        report.append(f"Threshold: {analysis.get('threshold', 'N/A')}")
        report.append(f"\n>>> VERDICT: {analysis.get('final_verdict', 'N/A')} <<<")

        return "\n".join(report)


## MAIN EXECUTION: RUN HERE!
some example prompts...
- Ticker: KXPRESNOMR-28-JDV, Race ID: 2028_REP_PRES_NOM
- Ticker: KXTENNCOACH-26-MMCC, Race ID: COACH_26
- Ticker: KXPRESNOMD-28-GN, Race ID: 2028_DEM_PRES_NOM
- Ticker: KXSENATETXR-26-KP, Race ID: 2026_REP_NOM _TX-SENATE
- Ticker: KXSENATETXD-26-JCRO, Race ID: 2026_DEM_NOM _TX-SENATE



In [ ]:
#                    MAIN EXECUTION BLOCK
#              Run this to test all three techniques

if __name__ == "__main__":
    """
    ┌─────────────────────────────────────────────────────────┐
    │                    MAIN EXECUTION                       │
    │         Tests All Three Techniques Together             │
    └─────────────────────────────────────────────────────────┘

    BEFORE RUNNING:
    1. Set your API key in the notebook:
       AZURE_API_KEY = "5aa2f6d151e04a2a92023c0d0a74b355"

    2. Make sure polls.csv exists with required columns:
       - race_id
       - snapshot_date
       - candidate1_support_pct
    """


    # Configuration
    example_ticker = "KXSENATETXR-26-KP"  # Replace with your market
    example_race = "2026_REP_NOM _TX-SENATE"            # Replace with your race_id
    polling_csv_path = "polls.csv"          # Your polling data file


    # Initialize the full system
    hunter = BiasHunterWithDataOps(polling_csv_path)

    # Run full analysis with all three techniques
    full_analysis = hunter.full_analysis_with_dataops(
        market_ticker=example_ticker,
        race_id=example_race,
        threshold=0.10,
    )

    # Generate and print the report
    report = hunter.generate_report(full_analysis)
    print("\n" + report)



                    BIAS HUNTER ANALYSIS REPORT
              RAG + AGENT + AUTOMATED LLM DATAOPS

Market Ticker: KXSENATETXR-26-KP
Race ID: 2026_REP_NOM _TX-SENATE
Threshold: 0.1

----------------------------------------------------------------------
 TECHNIQUE 1: RAG (Retrieval-Augmented Generation)
----------------------------------------------------------------------
Poll-Implied Probability: 0.29800000000000004
Number of Polls Retrieved: 5
Evidence Summary:
  1. Co/efficient: 28%
  2. University of Houston: 33%
  3. Emerson: 30%

----------------------------------------------------------------------
 INTERMEDIATE: AGENT (Kalshi Market Tool)
----------------------------------------------------------------------
Market Title: Wil Ken Paxton be the Republican nominee for the Senate in Texas?
Market Probability: 0.57
Trading Volume: 242221

----------------------------------------------------------------------
  TECHNIQUE 2a: DATAOPS (LLM Labeler)
------------------------------------

## BONUS*: output final verdict in a dashboard


In [ ]:
from IPython.display import HTML, display
import json

html_template = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Bias Hunter Interactive Analysis</title>

    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }

        body {
            font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            padding: 20px;
            min-height: 100vh;
        }

        .container { max-width: 1400px; margin: auto; }

        .header {
            background: white; border-radius: 20px; padding: 40px; margin-bottom: 30px;
            box-shadow: 0 10px 40px rgba(0,0,0,0.1); text-align: center;
        }

        .header h1 {
            font-size: 3em;
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            -webkit-background-clip: text; -webkit-text-fill-color: transparent;
        }

        .input-section {
            background: white; border-radius: 15px; padding: 30px; margin-bottom: 30px;
            box-shadow: 0 5px 20px rgba(0,0,0,0.1);
        }

        .input-group { margin-bottom: 20px; }
        .input-group label { display:block; font-weight:600; margin-bottom:5px; color:#444; }

        .input-group input {
            width:100%; padding:12px; border:2px solid #ddd; border-radius:8px;
            font-size:1em; transition:0.25s;
        }

        .input-group input:focus { border-color: #667eea; outline:none; }

        .btn {
            padding:15px 30px; border:none; border-radius:10px; cursor:pointer;
            font-weight:600; font-size:1.1em; transition:0.3s;
        }

        .btn-primary {
            background: linear-gradient(135deg,#667eea,#764ba2); color:white;
            flex:1;
        }
        .btn-primary:hover { transform:translateY(-2px); }

        .btn-secondary {
            background:#f8f9fa; border:2px solid #667eea; color:#667eea;
        }
        .btn-secondary:hover { background:#667eea; color:white; }

        .button-group { display:flex; gap:15px; margin-top:20px; }

        .loading { text-align:center; display:none; color:white; }
        .loading.active { display:block; }
        .spinner {
            border:4px solid rgba(255,255,255,0.3); border-top:4px solid white;
            border-radius:50%; width:50px; height:50px; margin:20px auto;
            animation: spin 1s linear infinite;
        }
        @keyframes spin { 100% { transform: rotate(360deg); } }

        .results { display:none; }
        .results.active { display:block; }

        .grid {
            display:grid; gap:20px;
            grid-template-columns: repeat(auto-fit, minmax(600px, 1fr));
        }

        .card {
            background:white; padding:30px; border-radius:15px;
            box-shadow:0 5px 20px rgba(0,0,0,0.1);
        }

        .card-title {
            font-size:1.5em; font-weight:700; color:#667eea;
            border-bottom:3px solid #667eea; padding-bottom:10px; margin-bottom:20px;
        }

        .metric {
            background:#f8f9fa; padding:10px 15px; margin:10px 0; border-radius:10px;
            display:flex; justify-content:space-between; font-size:1.05em;
        }

        .evidence, .insight {
            border-radius:10px; padding:15px; margin:12px 0; line-height:1.5;
        }
        .evidence { background:#e8f5e9; border-left:4px solid #4caf50; }
        .insight { background:#fff9e6; border-left:4px solid #ffc107; }

        .verdict-card {
            background: linear-gradient(135deg,#f093fb,#f5576c); color:white;
            padding:40px; border-radius:20px; text-align:center; margin-top:30px;
        }

        .verdict-card.insufficient {
            background: linear-gradient(135deg,#ff6b6b,#ee5a6f);
        }

        .error-message {
            display:none; background:#f8d7da; border-left:4px solid #c00;
            padding:20px; margin-top:20px; border-radius:10px; color:#721c24;
        }
        .error-message.active { display:block; }

        .warning-box {
            background:#fff3cd; border-left:4px solid #ffc107;
            padding:15px; margin:15px 0; border-radius:10px; color:#856404;
        }
    </style>
</head>

<body>
<div class="container">

    <div class="header">
        <h1> BIAS HUNTER ANALYSIS</h1>
        <div class="subtitle">RAG + AGENT + AUTOMATED LLM DATAOPS</div>
    </div>

    <div class="input-section">
        <h2 style="color: #667eea; margin-bottom: 20px;">Enter Analysis Parameters</h2>

        <div class="input-group">
            <label>Market Ticker</label>
            <input id="marketInput" placeholder="e.g., KXPRESNOMR-28-JDV" value="KXPRESNOMR-28-JDV">
        </div>

        <div class="input-group">
            <label>Race ID</label>
            <input id="raceInput" placeholder="e.g., 2028_REP_PRES_NOM" value="2028_REP_PRES_NOM">
        </div>

        <div class="input-group">
            <label>Threshold (0 to 1)</label>
            <input id="thresholdInput" type="number" step="0.01" min="0" max="1" placeholder="0.10" value="0.10">
        </div>

        <div class="input-group">
            <label>
                <input type="checkbox" id="simulateNoData" style="width:auto; display:inline-block; margin-right:5px;">
                Simulate Insufficient Data
            </label>
        </div>

        <div class="button-group">
            <button class="btn btn-primary" onclick="submitForm()">🚀 Run Analysis</button>
            <button class="btn btn-secondary" onclick="clearResults()">🔄 Clear</button>
        </div>
    </div>

    <div id="errorMessage" class="error-message"></div>

    <div id="loadingState" class="loading">
        <div class="spinner"></div>
        Running Bias Hunter analysis…
    </div>

    <div id="resultsSection" class="results">
        <div class="grid" id="mainGrid"></div>
        <div class="verdict-card" id="verdictCard"></div>
    </div>
</div>

<script>

function showError(msg) {
    const e = document.getElementById("errorMessage");
    e.textContent = "❌ " + msg;
    e.classList.add("active");
}

function clearResults() {
    document.getElementById("resultsSection").classList.remove("active");
    document.getElementById("errorMessage").classList.remove("active");
    document.getElementById("mainGrid").innerHTML = '';
    document.getElementById("verdictCard").innerHTML = '';
}

function submitForm() {
    const ticker = document.getElementById("marketInput").value.trim();
    const raceId = document.getElementById("raceInput").value.trim();
    const thresh = parseFloat(document.getElementById("thresholdInput").value);
    const simulateNoData = document.getElementById("simulateNoData").checked;

    if (!ticker || !raceId) return showError("Please enter Market Ticker AND Race ID.");
    if (isNaN(thresh) || thresh < 0 || thresh > 1) return showError("Threshold must be between 0 and 1.");

    runAnalysis(ticker, raceId, thresh, simulateNoData);
}

function generateMockAnalysis(ticker, raceId, threshold, simulateNoData = false) {
    // Simulate insufficient data scenario
    if (simulateNoData || Math.random() < 0.3) {
        // 30% chance of insufficient data, or forced by checkbox
        const missingMarket = simulateNoData || Math.random() < 0.5;
        const missingPolls = simulateNoData || Math.random() < 0.5;

        return {
            market_ticker: ticker,
            race_id: raceId,
            threshold: threshold,
            timestamp: new Date().toISOString(),
            rag_results: {
                technique: "RAG (Retrieval-Augmented Generation)",
                description: "Retrieved relevant polls and computed probability",
                poll_implied_prob: missingPolls ? null : 0.45,
                num_polls_retrieved: missingPolls ? 0 : 2,
                polling_evidence: missingPolls ? [] : [
                    { race_id: raceId, snapshot_date: "2024-10-01", pollster: "Mock Polls Inc.", candidate1_support_pct: "43.5" }
                ]
            },
            agent_results: missingMarket ? {
                technique: "AGENT (Tool-based)",
                error: `No market found for ${ticker}`
            } : {
                technique: "AGENT (Tool-based)",
                description: "Used Kalshi API tool to fetch live market data",
                market_ticker: ticker,
                title: "Mock Market for " + ticker,
                market_prob: 0.52,
                yes_price_pct: 52,
                volume: 150000
            },
            dataops_labels: {
                technique: "DATAOPS (LLM-based Labeling)",
                description: "LLM automatically labeled data with sentiment and risk flags",
                labels: {
                    sentiment: "uncertain",
                    time_horizon: "long_term",
                    confidence_level: "low",
                    risk_flags: ["stale_data", "low_volume"],
                    reasoning: missingPolls || missingMarket ?
                        "Insufficient data available for confident analysis." :
                        "Limited data quality affects reliability."
                }
            },
            dataops_judgment: missingMarket ? {
                technique: "DATAOPS (LLM-as-Judge)",
                error: "Cannot judge - missing market probability"
            } : {
                technique: "DATAOPS (LLM-as-Judge)",
                description: "LLM evaluated market efficiency vs polling baseline",
                judgment: {
                    efficiency_score: missingPolls ? "N/A" : "5.5",
                    verdict: "uncertain",
                    mispricing_direction: "uncertain",
                    confidence: 0.3,
                    key_insights: [
                        "Insufficient polling data to establish baseline",
                        "Market data available but lacks comparison reference"
                    ],
                    recommendation: "Wait for more polling data before making trading decisions."
                }
            },
            final_verdict: "INSUFFICIENT_DATA",
            discrepancy: null,
            discrepancy_pct: "N/A"
        };
    }

    // Normal case with sufficient data
    const marketProb = 0.3 + Math.random() * 0.4;
    const pollProb = marketProb + (Math.random() - 0.5) * 0.25;
    const discrepancy = Math.abs(marketProb - pollProb);

    let finalVerdict = "FAIRLY_PRICED";
    if (discrepancy > threshold) {
        finalVerdict = marketProb > pollProb ? "OVERPRICED" : "UNDERPRICED";
    }

    return {
        market_ticker: ticker,
        race_id: raceId,
        threshold: threshold,
        timestamp: new Date().toISOString(),
        rag_results: {
            technique: "RAG (Retrieval-Augmented Generation)",
            description: "Retrieved relevant polls and computed probability",
            poll_implied_prob: pollProb,
            num_polls_retrieved: 5,
            polling_evidence: [
                { race_id: raceId, snapshot_date: "2024-10-01", pollster: "Mock Polls Inc.", candidate1_support_pct: (pollProb*100 + Math.random()*10 - 5).toFixed(1) },
                { race_id: raceId, snapshot_date: "2024-09-15", pollster: "DataFakers LLC", candidate1_support_pct: (pollProb*100 + Math.random()*10 - 5).toFixed(1) }
            ]
        },
        agent_results: {
            technique: "AGENT (Tool-based)",
            description: "Used Kalshi API tool to fetch live market data",
            market_ticker: ticker,
            title: "Mock Market for " + ticker,
            market_prob: marketProb,
            yes_price_pct: (marketProb*100).toFixed(1),
            volume: Math.floor(Math.random() * 500000) + 50000
        },
        dataops_labels: {
            technique: "DATAOPS (LLM-based Labeling)",
            description: "LLM automatically labeled data with sentiment and risk flags",
            labels: {
                sentiment: "neutral",
                time_horizon: "near_term",
                confidence_level: "medium",
                risk_flags: ["polling_bias"],
                reasoning: "Mock data for demonstration."
            }
        },
        dataops_judgment: {
            technique: "DATAOPS (LLM-as-Judge)",
            description: "LLM evaluated market efficiency vs polling baseline",
            judgment: {
                efficiency_score: (10 - discrepancy * 10 / (0.25) * 5).toFixed(1),
                verdict: discrepancy > threshold ? "inefficient" : "efficient",
                mispricing_direction: finalVerdict.toLowerCase(),
                confidence: 0.7 + Math.random() * 0.3,
                key_insights: ["Market and poll probabilities show some variance.", "This is a mock analysis for demonstration purposes."],
                recommendation: "Consider actual data for real insights."
            }
        },
        final_verdict: finalVerdict,
        discrepancy: discrepancy,
        discrepancy_pct: `${(discrepancy * 100).toFixed(2)}%`
    };
}

function runAnalysis(ticker, raceId, threshold, simulateNoData = false) {
    document.getElementById("errorMessage").classList.remove("active");
    document.getElementById("loadingState").classList.add("active");
    document.getElementById("resultsSection").classList.remove("active");

    setTimeout(() => {
        const data = generateMockAnalysis(ticker, raceId, threshold, simulateNoData);
        displayResults(data);

        document.getElementById("loadingState").classList.remove("active");
        document.getElementById("resultsSection").classList.add("active");
    }, 1500);
}

function displayResults(a) {
    let rag = a.rag_results || {};
    let agent = a.agent_results || {};
    let dataops_labels = a.dataops_labels && a.dataops_labels.labels ? a.dataops_labels.labels : {};
    let dataops_judgment = a.dataops_judgment && a.dataops_judgment.judgment ? a.dataops_judgment.judgment : {};

    const isInsufficient = a.final_verdict === "INSUFFICIENT_DATA";

    let ragHtml = `
        <div class="card">
            <div class="card-title">📊 Technique 1: RAG</div>
            ${isInsufficient && rag.num_polls_retrieved === 0 ?
                '<div class="warning-box">⚠️ No polling data found for this race.</div>' : ''}
            <div class="metric"><span>Poll-Implied Probability</span><span>${rag.poll_implied_prob !== null ? (rag.poll_implied_prob * 100).toFixed(1) + '%' : 'N/A'}</span></div>
            <div class="metric"><span>Number of Polls Retrieved</span><span>${rag.num_polls_retrieved || 0}</span></div>
            ${rag.polling_evidence && rag.polling_evidence.length > 0 ?
                `<p style="margin-top:15px; font-weight:600;">Evidence Summary:</p>` +
                rag.polling_evidence.map((p,i)=>`
                    <div class="evidence"><strong>${i+1}. ${p.pollster || 'Unknown'}</strong><br>
                    Support: ${p.candidate1_support_pct || 'N/A'}% | Date: ${p.snapshot_date || 'N/A'}</div>
                `).join("") : '<p style="color:#999; font-style:italic;">No polling evidence available.</p>'
            }
        </div>
    `;

    let agentHtml = `
        <div class="card">
            <div class="card-title">🤖 Intermediate: Agent</div>
            ${agent.error ?
                `<div class="warning-box">⚠️ ${agent.error}</div>` : ''}
            <div class="metric"><span>Market Title</span><span>${agent.title || 'N/A'}</span></div>
            <div class="metric"><span>Market Probability</span><span>${agent.market_prob !== null && agent.market_prob !== undefined ? (agent.market_prob * 100).toFixed(1) + '%' : 'N/A'}</span></div>
            <div class="metric"><span>Trading Volume</span><span>${agent.volume || 'N/A'}</span></div>
            <div class="metric"><span>Calculated Discrepancy</span><span>${a.discrepancy_pct || 'N/A'}</span></div>
            <div class="metric"><span>Market Status</span><span>${a.final_verdict || 'N/A'}</span></div>
        </div>
    `;

    let dataopsLabelsHtml = `
        <div class="card">
            <div class="card-title">🏷️ Technique 2a: DataOps (LLM Labeler)</div>
            <div class="metric"><span>Sentiment</span><span>${dataops_labels.sentiment || 'N/A'}</span></div>
            <div class="metric"><span>Time Horizon</span><span>${dataops_labels.time_horizon || 'N/A'}</span></div>
            <div class="metric"><span>Confidence Level</span><span>${dataops_labels.confidence_level || 'N/A'}</span></div>
            <div class="metric"><span>Risk Flags</span><span>${(dataops_labels.risk_flags && dataops_labels.risk_flags.length > 0) ? dataops_labels.risk_flags.join(', ') : 'None'}</span></div>
            <p style="margin-top:15px; font-weight:600;">Reasoning:</p><div class="insight">${dataops_labels.reasoning || 'N/A'}</div>
        </div>
    `;

    let dataopsJudgeHtml = `
        <div class="card">
            <div class="card-title">⚖️ Technique 2b: DataOps (LLM Judge)</div>
            ${a.dataops_judgment && a.dataops_judgment.error ?
                `<div class="warning-box">⚠️ ${a.dataops_judgment.error}</div>` : ''}
            <div class="metric"><span>Efficiency Score</span><span>${dataops_judgment.efficiency_score !== undefined ? dataops_judgment.efficiency_score + '/10' : 'N/A'}</span></div>
            <div class="metric"><span>Verdict</span><span>${dataops_judgment.verdict || 'N/A'}</span></div>
            <div class="metric"><span>Mispricing Direction</span><span>${dataops_judgment.mispricing_direction || 'N/A'}</span></div>
            <div class="metric"><span>Confidence</span><span>${dataops_judgment.confidence !== undefined ? dataops_judgment.confidence : 'N/A'}</span></div>
            ${dataops_judgment.key_insights && dataops_judgment.key_insights.length > 0 ?
                `<p style="margin-top:15px; font-weight:600;">Key Insights:</p>` +
                dataops_judgment.key_insights.map(insight => `<div class="insight">• ${insight}</div>`).join("") : ''
            }
            ${dataops_judgment.recommendation ?
                `<p style="margin-top:15px; font-weight:600;">Recommendation:</p><div class="insight">${dataops_judgment.recommendation}</div>` : ''}
        </div>
    `;

    document.getElementById("mainGrid").innerHTML = ragHtml + agentHtml + dataopsLabelsHtml + dataopsJudgeHtml;

    const verdictCard = document.getElementById("verdictCard");
    if (isInsufficient) {
        verdictCard.className = "verdict-card insufficient";
    } else {
        verdictCard.className = "verdict-card";
    }

    verdictCard.innerHTML = `
        <h2>>>> VERDICT: ${a.final_verdict || 'INSUFFICIENT_DATA'} <<<</h2>
        ${isInsufficient ?
            '<p style="margin-top:15px; font-size:1.1em;">Unable to generate verdict due to missing data. Please verify ticker and race ID, or wait for more data availability.</p>' :
            `<p><strong>Discrepancy:</strong> ${a.discrepancy_pct || 'N/A'} |
            <strong>Threshold:</strong> ${a.threshold !== undefined ? (a.threshold * 100).toFixed(1) + '%' : 'N/A'}</p>`
        }
        <p>Market: ${a.market_ticker || 'N/A'} | Race: ${a.race_id || 'N/A'}</p>
    `;
}

document.addEventListener('DOMContentLoaded', () => {
    const defaultTicker = document.getElementById('marketInput').value;
    const defaultRaceId = document.getElementById('raceInput').value;
    const defaultThreshold = parseFloat(document.getElementById('thresholdInput').value);
    runAnalysis(defaultTicker, defaultRaceId, defaultThreshold, false);
});

</script>

</body>
</html>"""

display(HTML(html_template))